# Exploratory Data Analysis (EDA)
## Adaptive Explainable Predictive Maintenance

This notebook conducts exploratory data analysis on the raw Scania APS Failure dataset. We analyze:
1. Class Imbalance (Target distributions)
2. Missing Values (Nullness profiles across features)
3. Basic correlation structures

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.data.data_loader import load_raw_data
from src.data.data_validator import validate_raw_dataframe

### 1. Load Raw Datasets

In [ ]:
train_path = os.path.join("..", "data", "raw", "aps_failure_training_set.csv")
test_path = os.path.join("..", "data", "raw", "aps_failure_test_set.csv")

df_train = load_raw_data(train_path)
df_test = load_raw_data(test_path)

print(f"Training Set Shape: {df_train.shape}")
print(f"Test Set Shape: {df_test.shape}")

### 2. Verify Validation Checks

In [ ]:
validate_raw_dataframe(df_train, is_training=True)
validate_raw_dataframe(df_test, is_training=False)
print("Validation checks passed successfully!")

### 3. Analyze Target Class Distribution

In [ ]:
train_counts = df_train["class"].value_counts()
train_pct = df_train["class"].value_counts(normalize=True) * 100

print("Training Set Distribution:")
for idx in train_counts.index:
    print(f"  {idx}: {train_counts[idx]} ({train_pct[idx]:.2f}%)")

### 4. Missing Values Analysis

In [ ]:
null_counts = df_train.drop(columns=["class"]).isnull().sum()
null_ratios = null_counts / len(df_train)

print("Feature Nullness Statistics:")
print(null_ratios.describe())

threshold = 0.70
cols_to_drop = null_ratios[null_ratios > threshold].index.tolist()
print(f"\nNumber of columns with >{threshold*100}% missing values: {len(cols_to_drop)}")
print(f"Columns to drop: {cols_to_drop}")

### 5. Plot Null Ratio Distributions

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(null_ratios, bins=30, color='skyblue', edgecolor='black')
plt.axvline(x=threshold, color='red', linestyle='--', label=f'Threshold ({threshold:.0%})')
plt.title('Distribution of Feature Missing Value Ratios')
plt.xlabel('Missing Value Ratio')
plt.ylabel('Count of Features')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 6. Correlation of Features with Class Target

In [ ]:
# Map target labels pos/neg to numeric 1/0 for correlation calculation
y_numeric = df_train["class"].map({"neg": 0, "pos": 1})

# Impute missing values with median for a quick correlation check
df_numeric_features = df_train.drop(columns=["class"]).drop(columns=cols_to_drop)
df_imputed = df_numeric_features.fillna(df_numeric_features.median())

# Compute correlation with target
correlations = df_imputed.corrwith(y_numeric).abs().sort_values(ascending=False)
print("Top 10 features correlated with target class:")
print(correlations.head(10))